In [3]:
# ============================================================
# NEDT (National Energy Digital Twin) — RDF / Knowledge Graph Query Notebook
# ============================================================

from pathlib import Path
from rdflib import Graph
import pandas as pd

# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------
ontology_file = Path("DT_ontology.ttl")
instances_file = Path("DT_instances_v11.ttl")

# ------------------------------------------------------------
# 2. Safety checks
# ------------------------------------------------------------
def check_file_exists(file_path: Path):
    if not file_path.exists():
        raise FileNotFoundError(
            f"Could not find {file_path.name} in the current folder.\n"
            f"Current folder is: {Path.cwd()}"
        )

def check_not_rtf(file_path: Path):
    text = file_path.read_text(encoding="utf-8", errors="ignore")
    if text.lstrip().startswith(r"{\rtf"):
        raise ValueError(
            f"{file_path.name} is an RTF rich text file, not plain Turtle.\n"
            f"Open it in VS Code and save it again as plain text."
        )
    return text

check_file_exists(ontology_file)
check_file_exists(instances_file)

ontology_text = check_not_rtf(ontology_file)
instances_text = check_not_rtf(instances_file)

print("Files found and confirmed as plain text:")
print("-", ontology_file.name)
print("-", instances_file.name)

# ------------------------------------------------------------
# 3. Load RDF graph
# ------------------------------------------------------------
g = Graph()
g.parse(data=ontology_text, format="turtle")
g.parse(data=instances_text, format="turtle")

print(f"\nLoaded RDF graph successfully.")
print(f"Total triples: {len(g)}")

# ------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------
NEDT_PREFIX = "https://example.org/nedt#"

def shorten_uri(value):
    """
    Make RDF output more readable.
    Converts long URIs like https://example.org/nedt#dwelling_D001
    into nedt:dwelling_D001
    """
    value = str(value)
    if value.startswith(NEDT_PREFIX):
        return "nedt:" + value.replace(NEDT_PREFIX, "")
    return value

def run_query(graph: Graph, query: str, shorten=True) -> pd.DataFrame:
    """
    Run a SPARQL query and return results as a pandas DataFrame.
    """
    results = graph.query(query)
    rows = []
    for row in results:
        row_values = []
        for item in row:
            if item is None:
                row_values.append(None)
            else:
                row_values.append(shorten_uri(item) if shorten else str(item))
        rows.append(row_values)

    columns = [str(v) for v in results.vars]
    return pd.DataFrame(rows, columns=columns)

def print_title(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

# ------------------------------------------------------------
# 5. SPARQL queries
# ------------------------------------------------------------

# Query 1: all dwellings and counties
query_1 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?dwelling ?county
WHERE {
  ?dwelling a nedt:Dwelling ;
            nedt:locatedInCounty ?county .
}
ORDER BY ?dwelling
"""

# Query 2: dwellings and heating systems
query_2 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?dwelling ?heatingSystem
WHERE {
  ?dwelling a nedt:Dwelling ;
            nedt:usesHeatingSystem ?heatingSystem .
}
ORDER BY ?dwelling
"""

# Query 3: demand profiles with annual energy and peak demand
query_3 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?demandProfile ?annualEnergy ?peakDemand
WHERE {
  ?demandProfile a nedt:DemandProfile ;
                 nedt:hasAnnualEnergyUse ?annualEnergy ;
                 nedt:hasPeakDemandKW ?peakDemand .
}
ORDER BY ?demandProfile
"""

# Query 4: dwellings in Dublin with gas boilers
query_4 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?dwelling
WHERE {
  ?dwelling a nedt:Dwelling ;
            nedt:locatedInCounty nedt:county_Dublin ;
            nedt:usesHeatingSystem ?heating .
  ?heating a nedt:GasBoiler .
}
ORDER BY ?dwelling
"""

# Query 5: dwellings served by substation S33
query_5 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?dwelling
WHERE {
  ?dwelling a nedt:Dwelling ;
            nedt:isServedBy nedt:substation_S33 .
}
ORDER BY ?dwelling
"""

# Query 6: annual energy and peak demand by dwelling
query_6 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?dwelling ?annualEnergy ?peakDemand
WHERE {
  ?demandProfile a nedt:DemandProfile ;
                 nedt:forDwelling ?dwelling ;
                 nedt:hasAnnualEnergyUse ?annualEnergy ;
                 nedt:hasPeakDemandKW ?peakDemand .
}
ORDER BY ?dwelling
"""

# Query 7: dwellings, archetypes, and scenarios
query_7 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?dwelling ?archetype ?scenario
WHERE {
  ?dwelling a nedt:Dwelling ;
            nedt:instantiatesArchetype ?archetype ;
            nedt:evaluatedUnderScenario ?scenario .
}
ORDER BY ?dwelling
"""

# Query 8: observations with values and timestamps
query_8 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?observation ?space ?value ?time
WHERE {
  ?observation a nedt:Observation ;
               nedt:measuredInSpace ?space ;
               nedt:hasMeasuredValue ?value ;
               nedt:hasTimestamp ?time .
}
ORDER BY ?observation
"""

# Query 9: all scenarios and adoption rates
query_9 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?scenario ?year ?hpRate ?evRate ?v2hRate
WHERE {
  ?scenario a nedt:Scenario ;
            nedt:hasScenarioYear ?year ;
            nedt:hasHeatPumpAdoptionRate ?hpRate ;
            nedt:hasEVAdoptionRate ?evRate ;
            nedt:hasV2HAdoptionRate ?v2hRate .
}
ORDER BY ?scenario
"""

# Query 10: demand profiles validated against observations
query_10 = """
PREFIX nedt: <https://example.org/nedt#>

SELECT ?demandProfile ?validationObservation
WHERE {
  ?demandProfile a nedt:DemandProfile ;
                 nedt:validatedAgainstObservation ?validationObservation .
}
ORDER BY ?demandProfile
"""

# ------------------------------------------------------------
# 6. Run and display queries
# ------------------------------------------------------------
queries = {
    "Query 1: Dwellings and counties": query_1,
    "Query 2: Dwellings and heating systems": query_2,
    "Query 3: Demand profiles with energy and peak demand": query_3,
    "Query 4: Dwellings in Dublin with gas boilers": query_4,
    "Query 5: Dwellings served by substation S33": query_5,
    "Query 6: Annual energy and peak demand by dwelling": query_6,
    "Query 7: Dwellings, archetypes, and scenarios": query_7,
    "Query 8: Observations with values and timestamps": query_8,
    "Query 9: Scenarios and adoption rates": query_9,
    "Query 10: Validation links": query_10,
}

results_dict = {}

for title, q in queries.items():
    print_title(title)
    df = run_query(g, q)
    results_dict[title] = df
    display(df)

# ------------------------------------------------------------
# 7. Optional: save query results to CSV files
# ------------------------------------------------------------
save_csv = False   # Change to True if you want CSV exports

if save_csv:
    output_dir = Path("query_outputs")
    output_dir.mkdir(exist_ok=True)

    for title, df in results_dict.items():
        safe_name = title.lower().replace(":", "").replace(" ", "_")
        out_file = output_dir / f"{safe_name}.csv"
        df.to_csv(out_file, index=False)

    print(f"\nCSV files saved to: {output_dir.resolve()}")

# ------------------------------------------------------------
# 8. Optional: inspect some raw triples
# ------------------------------------------------------------
print_title("First 20 triples in the graph")
for i, triple in enumerate(g):
    print(tuple(shorten_uri(x) for x in triple))
    if i >= 19:
        break

Files found and confirmed as plain text:
- ontology.ttl
- instances.ttl

Loaded RDF graph successfully.
Total triples: 283

Query 1: Dwellings and counties


,dwelling,county
0,cdt:dwelling_D001,cdt:county_Dublin
1,cdt:dwelling_D002,cdt:county_Dublin
2,cdt:dwelling_D003,cdt:county_Louth



Query 2: Dwellings and heating systems


,dwelling,heatingSystem
0,cdt:dwelling_D001,cdt:gasboiler_001
1,cdt:dwelling_D002,cdt:heatpump_002
2,cdt:dwelling_D003,cdt:oilboiler_003



Query 3: Demand profiles with energy and peak demand


...


Query 9: Scenarios and adoption rates


,scenario,year,hpRate,evRate,v2hRate
0,cdt:scenario_SCN2030,2030,0.40,0.20,0.10



Query 10: Validation links


,demandProfile,validationObservation
0,cdt:demand_D001_SCN2030,cdt:valobs_D001_SCN2030



First 20 triples in the graph
('cdt:hasV2HCapability', 'http://www.w3.org/2000/01/rdf-schema#range', 'cdt:V2HCharger')
('cdt:demand_D002_SCN2030', 'cdt:evaluatedUnderScenario', 'cdt:scenario_SCN2030')
('cdt:obs_CO2_001', 'http://www.w3.org/2000/01/rdf-schema#label', 'CO2 observation 001')
('cdt:occprofile_D002', 'cdt:generatedByOccupancyModel', 'cdt:occmodel_BNN_v1')
('cdt:building_B003', 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'cdt:Building')
('cdt:nationalAgg_SCN2030', 'http://www.w3.org/2000/01/rdf-schema#label', 'National aggregate SCN2030')
('cdt:demand_D001_SCN2030', 'cdt:hasAnnualEnergyUse', '12500')
('cdt:occstate_D001_0800', 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'cdt:OccupancyState')
('cdt:archetype_A233', 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'cdt:Archetype')
('cdt:obs_CO2_001', 'cdt:hasMeasuredValue', '1120')
('cdt:scenario_SCN2030', 'http://www.w3.org/2000/01/rdf-schema#label', 'Scenario 2030')
('cdt:hasCapacityValue', 'http://www.w3.or